In [6]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.1 MB/s eta 0:00:00


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import random
import shutil

# paths
images_path = r"/content/drive/MyDrive/NumberPlate_Datset_updated/NumberPlate Datset with annotation/images"
labels_path = r"/content/drive/MyDrive/NumberPlate_Datset_updated/NumberPlate Datset with annotation/yolo_labels"

train_img_path = r"D:\NumberPlate Datset with annotation\images\train"
test_img_path = r"D:\NumberPlate Datset with annotation\images\test"
train_label_path = r"D:\NumberPlate Datset with annotation\yolo_labels\train"
test_label_path = r"D:\NumberPlate Datset with annotation\yolo_labels\test"

# create new directories
os.makedirs(train_img_path, exist_ok=True)
os.makedirs(test_img_path, exist_ok=True)
os.makedirs(train_label_path, exist_ok=True)
os.makedirs(test_label_path, exist_ok=True)

# collect images
image_files = [f for f in os.listdir(images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# random shuffle
random.shuffle(image_files)

# split ratio
train_ratio = 0.8
split_index = int(len(image_files) * train_ratio)

train_files = image_files[:split_index]
test_files = image_files[split_index:]

def move_pair(file_list, img_dest, lbl_dest):
    for img_file in file_list:
        img_name, _ = os.path.splitext(img_file)
        lbl_file = img_name + ".txt"

        # move image
        shutil.move(os.path.join(images_path, img_file), os.path.join(img_dest, img_file))

        # move label if exists
        lbl_path_full = os.path.join(labels_path, lbl_file)
        if os.path.exists(lbl_path_full):
            shutil.move(lbl_path_full, os.path.join(lbl_dest, lbl_file))
        else:
            print(f"⚠ No label for: {img_file} (moved image only)")

move_pair(train_files, train_img_path, train_label_path)
move_pair(test_files, test_img_path, test_label_path)

print("🎉 Train-Test Split Completed Successfully!")

🎉 Train-Test Split Completed Successfully!


In [ ]:
#moving unmatched labels and images
import os
import shutil

# Paths
images_path = r"D:\NumberPlate Datset with annotation\images"
labels_path = r"D:\NumberPlate Datset with annotation\yolo_labels"

# Output folders
unlabeled_images_path = r"D:\NumberPlate Datset with annotation\unlabeled_images"
unmatched_labels_path = r"D:\NumberPlate Datset with annotation\unmatched_labels"

# Create output directories if not exists
os.makedirs(unlabeled_images_path, exist_ok=True)
os.makedirs(unmatched_labels_path, exist_ok=True)

# Get image and label file names
image_files = {os.path.splitext(f)[0] for f in os.listdir(images_path)}
label_files = {os.path.splitext(f)[0] for f in os.listdir(labels_path)}

# Find images without labels
unlabeled_images = image_files - label_files

# Find labels without images (optional)
unmatched_labels = label_files - image_files

# Move unlabeled images
for img_name in unlabeled_images:
    for ext in [".jpg", ".png", ".jpeg"]:
        img_file = os.path.join(images_path, img_name + ext)
        if os.path.exists(img_file):
            shutil.move(img_file, os.path.join(unlabeled_images_path, os.path.basename(img_file)))
            print(f"Moved Image Without Label: {img_file}")
            break

# Move unmatched labels (optional)
for lbl_name in unmatched_labels:
    lbl_file = os.path.join(labels_path, lbl_name + ".txt")
    if os.path.exists(lbl_file):
        shutil.move(lbl_file, os.path.join(unmatched_labels_path, os.path.basename(lbl_file)))
        print(f"Moved Label Without Image: {lbl_file}")

print("Done!")

In [3]:
# package import
import torch
import pandas as pd
import numpy as np
from PIL import Image
import yaml
from pathlib import Path
from os.path import sameopenfile

In [ ]:
import os
import yaml

import os
import yaml

# Define YOLO dataset parameters
yolo_parameters = {
    "path": "/content/yolov5/content/yolov5/guvi_data/image/train",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {
        0: "number_plate"
    }
}

# Ensure directory exists
os.makedirs("/content/yolov5/content/yolov5/guvi_data/image/train", exist_ok=True)

# Path to save YAML
path = "/content/yolov5/content/yolov5/guvi_data/train.yaml"

# Write YAML file
with open(path, "w") as f:
    yaml.dump(yolo_parameters, f)

print("numberplate.yaml created successfully!")



train.yaml created successfully!


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # start with nano model
model.train(data="/content/yolov5/content/yolov5/guvi_data/numberplate.yaml", epochs=50, imgsz=640, batch=16)


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolov5/content/yolov5/guvi_data/train.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective

RuntimeError: Dataset '/content/yolov5/content/yolov5/guvi_data/train.yaml' error ❌ Dataset '/content/yolov5/content/yolov5/guvi_data/train.yaml' images not found, missing path '/content/yolov5/content/yolov5/guvi_data/image/train/valid/images'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

In [12]:
from ultralytics import YOLO

# Load pretrained YOLO model (nano version = fastest, best for small datasets)
model = YOLO("yolov8n.pt")

# Train
model.train(
    data="/content/numberplate.yaml",
    epochs=50,
    batch=8,
    imgsz=640,
    workers=4,
    device=0   # use GPU if available, else CPU
)

print("Training Completed Successfully!")

Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/numberplate.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, p

In [14]:
from google.colab import files
files.download('runs/detect')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>